# Week 13 — Dataset A: OSF PsyArXiv

**用途**：在 VS Code / Jupyter 中一格一格 (cell-by-cell) 跑完整個 pipeline。

**對應 .py 檔**：`osf_psyarxiv_pipeline.py`

**在 PowerShell 中跑這個檔案**：
```powershell
# 在 VS Code 中：開啟此 .ipynb，右上角選 Python 直譯器，然後 Shift+Enter 逐格執行
# 在 PowerShell 中：
ipython -c "%run osf_psyarxiv_pipeline.ipynb"
```

**建議**：在 VS Code 中 cell-by-cell 跑，看每一段輸出。

## 1. 載入需要的套件

四個套件：
- `requests`：發 HTTP request 抓 API 資料
- `json`：把 API 回傳的 JSON 字串 parse 成 Python dict / list（也可反向 dump 出來檢視）
- `pandas`：把資料整理成表格 (DataFrame)
- `plotly.express`：畫互動圖表

> **注意**：`requests` 內建的 `r.json()` 已經會幫我們把回應轉成 Python 物件 — 為什麼還要 `import json`？因為 API 回來的結構通常是 **巢狀 dict**，肉眼很難讀。用 `json.dumps(obj, indent=2, ensure_ascii=False)` 可以把它印成「縮排排版好」的字串，這是 debug API 時最常用的小技巧。

In [1]:
import json
import time

import pandas as pd
import plotly.express as px
import requests

# 設定 OSF PsyArXiv 的 API endpoint
OSF_ENDPOINT = "https://api.osf.io/v2/preprint_providers/psyarxiv/preprints/"
HEADERS = {"User-Agent": "NS5116-week13-teaching-example"}

print("套件載入完成")

套件載入完成


## 2. 抓一頁試試看

在大量抓取之前，先抓 **一頁** 確認 API 回什麼樣子的資料。

`page[size]=100` 是 OSF 容許的單頁最大筆數。

In [2]:
params = {
    "page": 1,
    "page[size]": 100,
    "sort": "-date_published",   # 負號代表從新到舊
}
r = requests.get(OSF_ENDPOINT, params=params, timeout=30, headers=HEADERS)
r.raise_for_status()

# r.json() 內部會呼叫 json.loads(r.text)，把回應的 JSON 字串 → Python dict
data = r.json()

# 用 json.dumps() 把第一筆 preprint 印成縮排排版好的樣子，方便肉眼檢視 schema
sample = data["data"][0]
pretty = json.dumps(sample, indent=2, ensure_ascii=False)
print(pretty[:800], "\n...(truncated)")

# 同樣的資訊用 dict 操作抓重點欄位
print("\n頂層 keys：", list(sample.keys()))
print("標題：", sample["attributes"]["title"][:80])
print("發表日：", sample["attributes"]["date_published"])
print("tags 數量：", len(sample["attributes"].get("tags", [])))

{
  "id": "5jmf2_v1",
  "type": "preprints",
  "attributes": {
    "date_created": "2026-05-09T13:09:10.300348",
    "date_modified": "2026-05-13T13:31:21.595482",
    "date_published": "2026-05-13T13:31:21.558428",
    "original_publication_date": null,
    "custom_publication_citation": null,
    "doi": "",
    "title": "Threat Crowds Out Solidarity: How Politicization Erodes Discursive Support for Protest",
    "description": "Classical theories of protest reception, best articulated in Turner (1969), posit that public support emerges from competition among simultaneously active framing mechanisms whose dominance shifts as a conflict matures. Such claims have been hard to test: surveys are too sparse, static content analysis obscures dynamics, and experimental vignettes isolate single f 
...(truncated)

頂層 keys： ['id', 'type', 'attributes', 'relationships', 'links']
標題： Threat Crowds Out Solidarity: How Politicization Erodes Discursive Support for P
發表日： 2026-05-13T13:31:21.558428
t

## 3. 抓 10 頁 = 約 1,000 筆 (pagination)

**Pagination（分頁）**：就像 Google 搜尋結果一頁只給 10 個，要看更多必須點下一頁。OSF 也一樣，一次最多 100 筆。

我們 loop 從第 1 頁到第 10 頁，每頁之間 `time.sleep(0.3)` — 對 server 禮貌一點，免得被擋。

In [3]:
all_items = []
for page in range(1, 11):
    params = {"page": page, "page[size]": 100, "sort": "-date_published"}
    r = requests.get(OSF_ENDPOINT, params=params, timeout=30, headers=HEADERS)
    r.raise_for_status()
    batch = r.json().get("data", [])
    all_items.extend(batch)
    print(f"  page {page:2d}: 抓到 {len(batch):3d} 筆 (累計 {len(all_items)})")
    time.sleep(0.3)

print(f"\n總共抓到 {len(all_items)} 筆 preprint。")

  page  1: 抓到 100 筆 (累計 100)
  page  2: 抓到 100 筆 (累計 200)
  page  3: 抓到 100 筆 (累計 300)
  page  4: 抓到 100 筆 (累計 400)
  page  5: 抓到 100 筆 (累計 500)
  page  6: 抓到 100 筆 (累計 600)
  page  7: 抓到 100 筆 (累計 700)
  page  8: 抓到 100 筆 (累計 800)
  page  9: 抓到 100 筆 (累計 900)
  page 10: 抓到 100 筆 (累計 1000)

總共抓到 1000 筆 preprint。


## 4. 把 JSON 攤平成表格 (DataFrame)

回傳的 JSON 是 **巢狀 (nested)** 結構 — 像 Russian doll 套娃娃。
`subjects` 還是「list of list」(分類路徑)。

我們要把這種樹狀結構，攤平成「一列 = 一篇 preprint」的 DataFrame。

In [4]:
records = []
for it in all_items:
    a = it.get("attributes", {})
    subjects = a.get("subjects", [])
    # 每個 subject 路徑取最後一層 (最具體的分類)
    leaf_subjects = [chain[-1]["text"] for chain in subjects if chain]
    records.append({
        "id": it.get("id"),
        "title": (a.get("title") or "").strip(),
        "date_published": a.get("date_published"),
        "n_tags": len(a.get("tags", [])),
        "primary_subject": leaf_subjects[0] if leaf_subjects else None,
        "n_description_chars": len(a.get("description") or ""),
    })

df_raw = pd.DataFrame(records)
df_raw.head()

,id,title,date_published,n_tags,primary_subject,n_description_chars
0,5jmf2_v1,Threat Crowds Out Solidarity: How Politicizati...,2026-05-13T13:31:21.558428,14,Quantitative Methods,1243
1,w4ea3_v2,Afterword: Coping through crisis with coronamusic,2026-05-13T13:30:15.269700,8,Affect and Emotion Regulation,1143
2,f94p8_v1,Growing up in an Age of Economic Inequality: A...,2026-05-13T13:29:17.313141,1,Psychiatry,1026
3,c9xua_v3,Information-Based Sequential Monitoring in Lin...,2026-05-13T13:28:49.029326,4,Statistical Methods,1607
4,u3p8c_v2,The Group Development Questionnaire Short (GDQ...,2026-05-13T13:28:23.496619,4,Workgroup and Teams,867


## 5. 清理資料 — 「觀察 → 動作 → 代價」

Week 12 學的紀律：每個 cleaning 動作都要說得出 (1) 觀察到什麼問題、(2) 做了什麼動作、(3) 動作的代價是什麼。

本次需要做三件事：
1. `date_published` 是字串 → 轉成 datetime
2. `title` 可能是空字串 → 過濾掉
3. `primary_subject` 缺值 → 填 'Unspecified'

In [5]:
df = df_raw.copy()

# 1. 字串 → datetime
df["date_published"] = pd.to_datetime(df["date_published"], errors="coerce", utc=True)
df["year_month"] = (df["date_published"].dt.tz_convert(None)
                      .dt.to_period("M").dt.to_timestamp())

# 2. 空 title 過濾
df["title_len"] = df["title"].str.len()
df = df[df["title_len"] > 0].copy()

# 3. subject 缺值補字串
df["primary_subject"] = df["primary_subject"].fillna("Unspecified")

print(f"清理前: {len(df_raw)} 列 | 清理後: {len(df)} 列")
df.head()

清理前: 1000 列 | 清理後: 1000 列


,id,title,date_published,n_tags,primary_subject,n_description_chars,year_month,title_len
0,5jmf2_v1,Threat Crowds Out Solidarity: How Politicizati...,2026-05-13 13:31:21.558428+00:00,14,Quantitative Methods,1243,2026-05-01,86
1,w4ea3_v2,Afterword: Coping through crisis with coronamusic,2026-05-13 13:30:15.269700+00:00,8,Affect and Emotion Regulation,1143,2026-05-01,49
2,f94p8_v1,Growing up in an Age of Economic Inequality: A...,2026-05-13 13:29:17.313141+00:00,1,Psychiatry,1026,2026-05-01,87
3,c9xua_v3,Information-Based Sequential Monitoring in Lin...,2026-05-13 13:28:49.029326+00:00,4,Statistical Methods,1607,2026-05-01,56
4,u3p8c_v2,The Group Development Questionnaire Short (GDQ...,2026-05-13 13:28:23.496619+00:00,4,Workgroup and Teams,867,2026-05-01,116


## 6. 描述性統計 — 資料健康檢查

在畫圖之前，先用 `describe()` / `value_counts()` 看資料整體狀況。

In [6]:
print(f"列數          : {len(df)}")
print(f"日期範圍       : {df['date_published'].min().strftime('%Y-%m-%d')}"
      f"  →  {df['date_published'].max().strftime('%Y-%m-%d')}")
print(f"unique subjects: {df['primary_subject'].nunique()}")
print(f"n_tags M/SD    : {df['n_tags'].mean():.2f} / {df['n_tags'].std():.2f}")
print(f"title 長度 M/SD : {df['title_len'].mean():.0f} / {df['title_len'].std():.0f}")
print("\n前 8 大 subject:")
print(df["primary_subject"].value_counts().head(8).to_string())

列數          : 1000
日期範圍       : 2026-04-19  →  2026-05-13
unique subjects: 125
n_tags M/SD    : 3.46 / 3.54
title 長度 M/SD : 97 / 32

前 8 大 subject:
primary_subject
Social and Behavioral Sciences       112
Psychiatry                            97
Cognitive Neuroscience                64
Social and Personality Psychology     47
Meta-science                          37
Cognitive Psychology                  37
Clinical Psychology                   34
Developmental Psychology              33


## 7. 圖 1 — Top subjects bar chart

回答：「最近一千篇 PsyArXiv 文章，最熱門的主題是哪些？」

用水平 bar chart (`orientation="h"`)，因為 subject 名稱很長，放 y 軸比較不會擠。

In [7]:
counts = (df["primary_subject"].value_counts().head(15)
            .reset_index())
counts.columns = ["subject", "n"]

fig = px.bar(
    counts, x="n", y="subject", orientation="h",
    color="n", color_continuous_scale="Blues",
    title="Top 15 PsyArXiv subjects (recent ~1000 preprints)",
    labels={"n": "Number of preprints", "subject": "Primary subject"},
)
fig.update_layout(yaxis={"categoryorder": "total ascending"},
                  coloraxis_showscale=False)
fig.show()

## 8. 圖 2 — 每月發表量 (line + annotation)

回答：「最近這段時間哪個月發最多？」

用 `add_annotation` 在最高點加一個箭頭 — annotation 的重點是 **解釋為什麼這點重要**，不是只標數字。

In [8]:
monthly = (df.dropna(subset=["year_month"])
             .groupby("year_month").size()
             .reset_index(name="n_preprints"))

fig = px.line(monthly, x="year_month", y="n_preprints", markers=True,
              title="PsyArXiv preprints per month",
              labels={"year_month": "Month", "n_preprints": "Number of preprints"})

peak = monthly.loc[monthly["n_preprints"].idxmax()]
fig.add_annotation(
    x=peak["year_month"], y=peak["n_preprints"],
    text=f"Peak: {int(peak['n_preprints'])} preprints",
    showarrow=True, arrowhead=2,
    bgcolor="rgba(255,255,255,0.8)", bordercolor="black",
)
fig.update_layout(hovermode="x unified")
fig.show()

## 9. 圖 3 — Scatter: 標題長度 vs. tag 數量

想看「標題長的論文是不是 tag 也比較多？」

Scatter 是 EDA (探索性資料分析) 最常用的圖。`hover_name="title"` 讓滑鼠移過去就看到具體論文標題。

In [9]:
fig = px.scatter(
    df, x="n_tags", y="title_len",
    color="primary_subject",
    hover_name="title",
    hover_data={"date_published": True, "primary_subject": True},
    opacity=0.6,
    title="Title length vs. tag count",
    labels={"n_tags": "Number of tags", "title_len": "Title length (chars)"},
)
fig.update_layout(showlegend=False)   # subject 太多，legend 會擋畫面
fig.show()

## 10. 儲存清理後資料

存成 CSV，下次可以直接 `pd.read_csv()` 讀回來，不用再次呼叫 API。

In [10]:
df.to_csv("psyarxiv_recent.csv", index=False, encoding="utf-8-sig")
print("已存到 psyarxiv_recent.csv")

已存到 psyarxiv_recent.csv
